# Data sources

The main data source is the publication by [Sayaman et al.](https://doi.org/10.1016/j.immuni.2021.01.011).

The key datasets can be found here:  https://gdc.cancer.gov/about-data/publications/CCG-AIM-2020 (HRC Imputed Genotyping Data). Download all `chr_*.zip` and store their contents in `data/` directory (`*.dose.vcf.gz` files go into `data/imputed` while `*.info.gz` go into `data/info`). Note that these datasets are controlled and that you will need to obtain the appropriate permission via dbGaP. You might also find scripts from https://github.com/rwsayaman/TCGA_PanCancer_Immune_Genetics useful.

Other files can be fetched as follows:


In [ ]:
%%bash
# data: stores all input files
# results/out: output prefix where all analysis files will reside
mkdir -p data results/out

# TCGA IDs used in Sayaman et al.
curl -L https://api.gdc.cancer.gov/data/9df36037-3f9a-47aa-bcab-1c5554bb0443 | awk 'OFS="\t" {print $1"_"$1,$2"_"$2,$3,$4}' | sort > data/tcga-ids.txt
# UCSF PCA Ancestry (from Sayaman et al.)
curl -L https://api.gdc.cancer.gov/data/fdfa536a-c3c8-405d-99d9-bc9375b5084c | cut -d, -f2,6 | tr -d '"' | awk -F, '{print $1,$1 >> "data/ucsf-"tolower($2)".txt"}'
# MAGMA's gene location file (hg19)
curl -L https://raw.githubusercontent.com/endeneon/MAGMA_analysis_protocol/refs/heads/main/data_files/Rev.NCBI37.3.gene.loc | awk '{print $6,$2,$3,$4,$5,$1}' OFS="\t" > data/NCBI37.3.gene.loc

Now, let's query the GDC database to get the project ID, sex and the age of each TCGA sample used in this analysis.
Ages are normalized and reported as z-scores.

In [2]:
import requests, json
import pandas as pd

# Get the list of TCGA IDs.
with open("data/tcga-ids.txt") as f:
    samples = list({l.split()[2] for l in f})

# Fetch all metadata from GDC in batches via GDC REST API.
hits = []
step = 300  # how many records to fetch per request; currently, API supports up to ~300 per request
for start in range(0, len(samples) * 10, step):
    d = json.loads(
        requests.get("https://api.gdc.cancer.gov/cases", params={
            "filters": json.dumps({
                "op": "in",
                "content": {
                    "field": "cases.submitter_id",
                    "value": samples[start:start + step]
                }
            }),
            "fields": ",".join([
                "submitter_id",
                "project.project_id",
                "demographic.gender",
                "demographic.age_at_index",
                "demographic.age_is_obfuscated",
            ]),
            "format": "JSON",
            "size": f"{step}",
        }).content.decode())
    hits += d["data"]["hits"]
    assert d["data"]["pagination"]["pages"] == 1
    start += step
    if start >= len(samples): break
print(f"fetching done; hits={len(hits)}")

# Form the dataframe and apply z-score normalization. Remove samples with no reported age.
df = pd.DataFrame(
    sorted([
        (d["submitter_id"], d["project"]["project_id"],
            d.get("demographic", {}).get("gender", 'X')[0].upper(),
            d.get("demographic", {}).get("age_at_index", None))
        for d in hits
    ]),
    columns=["sample", "type", "sex", "age"]
)
df.sex = df.sex.apply(lambda x: {'M': 1, 'F': 2}.get(x, 0))
df = df[~df.age.isna()]
df.age = (df.age - df.age.mean()) / df.age.std()
df[['sample', 'age', 'type', 'sex']].to_csv("out/tcga-covariates.csv", index=False, header=False, sep='\t')

fetching done; hits=10128


Also, let's prepare the phenotype and pathway data used for GWAS and GSA later on from the RNA-seq expression data as follows:

In [ ]:
%%bash
curl -L https://api.gdc.cancer.gov/data/3586c0da-64d0-4b74-a449-5ff4d9136611 > data/EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv
Rscript phenotypes.r data/EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv data/pathways.txt
    # Takes ~30min

# Initialization

Here, we assume that the following tools are installed: 
- plink v2 (`plink2`)
- bcftools

All R packages are provided (and automatically installed) in `phenotypes.r`.

# Data preparation

Here we filter imputed TCGA VCFs (with $R^2 \geq 0.5$ filter) and convert them to plink v2 format.

In [ ]:
%%bash

# Step 1 (section 2.1.1): R2 filter (R2 >= 0.5); takes ~3 hours
mkdir -p results/out/vcf
parallel --plus 'bcftools view -i "R2>=0.5" {} | bgzip -c > results/out/vcf/{/...}.vcf.gz' ::: data/imputed/*.dose.vcf.gz
parallel 'tabix -p vcf {}' ::: results/out/vcf/*.vcf.gz

# Step 2: Convert to plink format; takes ~20 min
mkdir -p results/out/tmp
parallel --plus 'plink2 --double-id --vcf {} --make-bed --out results/out/tmp/{/..}' ::: results/out/vcf/*.vcf.gz
# SNP ID for the PLINK bim file
parallel "awk 'OFS=\"\t\" {ref=\$5; alt=\$6; if (ref>alt) { ref=\$6; alt=\$5 } print \$1,\$1\":\"\$4\":\"ref\":\"alt,\$3,\$4,\$5,\$6}' {} > results/out/tmp/{/}.updated" ::: results/out/tmp/chr*.bim
# Update IIDs and FIDs in the PLINK fam file from Birdseed file IDs to TCGA patient IDs
parallel 'plink2 --bfile {.} --bim {}.updated --update-ids data/tcga-ids.txt --make-pgen --out results/out/vcf/{/.}' ::: results/out/tmp/*.bim
# Merge all chromosomes in plink
plink2 --pmerge-list <(seq -f "results/out/vcf/chr%g" 1 22) --out results/out/vcf/hg19-merged

# Step 3: Annotate SNPs with dbSNP v155 (hg19); takes ~30 min
plink2 --pfile results/out/vcf/hg19-merged --make-bed --out results/out/vcf/hg19-merged
curl -L https://ftp.ncbi.nih.gov/snp/redesign/archive/b155/VCF/GCF_000001405.25.gz | gzip -c -d | \
    awk '{print $1"_"$2,$3}' | sed -e 's/NC_000001.10/1/;s/NC_000002.11/2/;s/NC_000003.11/3/;s/NC_000004.11/4/;s/NC_000005.9/5/;s/NC_000006.11/6/;s/NC_000007.13/7/;s/NC_000008.10/8/;s/NC_000009.11/9/;s/NC_000010.10/10/;s/NC_000011.9/11/;s/NC_000012.11/12/;s/NC_000013.10/13/;s/NC_000014.8/14/;s/NC_000015.9/15/;s/NC_000016.9/16/;s/NC_000017.10/17/;s/NC_000018.9/18/;s/NC_000019.9/19/;s/NC_000020.10/20/;s/NC_000021.8/21/;s/NC_000022.10/22/;s/NW_004775435.1/25/' \
    > data/dbsnp-v155-hg19.txt
awk '{print $1"_"$4, $2}' results/out/vcf/hg19.bim |\
    awk 'FNR==NR {a[$1]=$2; next} ($1 in a) {print $1, $2, a[$1]}' data/dbsnp-v155-hg19.txt - |\
    awk '{print $2, $2":"$3}' > results/out/vcf/hg19-merged.bim.annotated
plink2 --pfile results/out/vcf/hg19-merged --update-name results/out/vcf/hg19-merged.bim.annotated --make-pgen --out results/out/vcf/hg19

# Processing and GWAS

Here we perform necessary preprocessing steps:
- SNP MAF filter
- LD pruning (optional; one version has no LD pruning while other two use $r^2$ thresholds of 0.15 and 0.25)
- Closely related individuals in the target data may lead to overfitted results, limiting the generalisability of the results.
  Individuals that have a first or second degree relative in the sample ($π > 0.125$) are removed.
- PCA and final QC

In [ ]:
%%bash

# Step 1: use samples from a single population (here, EUR population; same goes for AFR, AMR and ASIAN); takes ~1 min
mkdir -p results/out/eur
plink2 --pfile results/out/vcf/hg19 --keep data/ucsf-eur.txt --out results/out/eur/hg19 --make-pgen

# Step 2 (section 2.1.1): Apply quality control to the subset (MAF < 0.005); takes ~1 min
plink2 --pfile results/out/eur/hg19 --write-snplist --out results/out/eur/hg19 --maf 0.005 --geno 0.01 --mind 0.01

# Step 3 (section 2.1.1): LD pruning to remove highly correlated SNPs (R2 > 0.25 and R2 > 0.15; window size = 200); takes ~1 min
#                         We also keep a version w/o LD pruning.
plink2 --pfile results/out/eur/hg19 --extract results/out/eur/hg19.snplist --out results/out/eur/hg19 --indep-pairwise 200 50 0.25
plink2 --pfile results/out/eur/hg19 --extract results/out/eur/hg19.snplist --out results/out/eur/hg19-lowr2 --indep-pairwise 200 50 0.15

# Step 4: Generate heterozygosity rates (HET) file; takes ~1 min
plink2 --pfile results/out/eur/hg19 --extract results/out/eur/hg19.prune.in --het --out results/out/eur/hg19
plink2 --pfile results/out/eur/hg19 --extract results/out/eur/hg19-lowr2.prune.in --het --out results/out/eur/hg19-lowr2
plink2 --pfile results/out/eur/hg19 --extract results/out/eur/hg19.snplist  --het --out results/out/eur/hg19-nold
Rscript correct_het.r results/out/eur/hg19.het results/out/eur/hg19.valid
Rscript correct_het.r results/out/eur/hg19-lowr2.het results/out/eur/hg19-lowr2.valid
Rscript correct_het.r results/out/eur/hg19-nold.het results/out/eur/hg19-nold.valid

# Step 5 (section 2.1.1): Calculate relatedness (𝜋 > 0.125); takes ~1 min
plink2 --pfile results/out/eur/hg19 --extract results/out/eur/hg19.prune.in --keep results/out/eur/hg19.valid --king-cutoff 0.125 --out results/out/eur/hg19
plink2 --pfile results/out/eur/hg19 --extract results/out/eur/hg19-lowr2.prune.in --keep results/out/eur/hg19-lowr2.valid --king-cutoff 0.125 --out results/out/eur/hg19-lowr2
plink2 --pfile results/out/eur/hg19 --keep results/out/eur/hg19-nold.valid --king-cutoff 0.125 --out results/out/eur/hg19-nold

# Step 6: Final QC; takes ~2 min
plink2 --pfile results/out/eur/hg19 --make-pgen --keep results/out/eur/hg19.king.cutoff.in.id --out results/out/eur/hg19-qc --extract results/out/eur/hg19.snplist
plink2 --pfile results/out/eur/hg19 --make-pgen --keep results/out/eur/hg19-lowr2.king.cutoff.in.id --out results/out/eur/hg19-lowr2-qc --extract results/out/eur/hg19.snplist
plink2 --pfile results/out/eur/hg19 --make-pgen --keep results/out/eur/hg19-nold.king.cutoff.in.id --out results/out/eur/hg19-nold-qc --extract results/out/eur/hg19.snplist
    # Takes ~2 min

# Step 7: PCA; takes ~15 min
plink2 --pfile results/out/eur/hg19-qc --pca 10 --out results/out/eur/hg19-qc
plink2 --pfile results/out/eur/hg19-lowr2-qc --pca 10 --out results/out/eur/hg19-lowr2-qc
plink2 --pfile results/out/eur/hg19-nold-qc --pca 10 --out results/out/eur/hg19-nold-qc
# Here, -f1-11 selects IDs and top 9 PCs (to account for at least 90% of the variation)
paste <(cut -f1-11 results/out/eur/hg19-qc.eigenvec | grep -Fwf <(cut -f1 results/out/tcga-covariates.csv)) \
      <(cut -f2- results/out/tcga-covariates.csv) | tr '\t' ' ' > results/out/eur/hg19-qc.tcga-covariates.csv
paste <(cut -f1-11 results/out/eur/hg19-lowr2-qc.eigenvec | grep -Fwf <(cut -f1 results/out/tcga-covariates.csv)) \
      <(cut -f2- results/out/tcga-covariates.csv) | tr '\t' ' ' > results/out/eur/hg19-lowr2-qc.tcga-covariates.csv
paste <(cut -f1-11 results/out/eur/hg19-nold-qc.eigenvec | grep -Fwf <(cut -f1 results/out/tcga-covariates.csv)) \
      <(cut -f2- results/out/tcga-covariates.csv) | tr '\t' ' ' > results/out/eur/hg19-nold-qc.tcga-covariates.csv

Now, let's run GWAS on the cleaned-up data:

In [ ]:
%%bash

# Step 8: Run GWAS association tests; takes ~1 hour per mode (~3 hours in total)
for mode in qc nold-qc lowr2-qc ; do
    plink2 --pfile results/out/eur/hg19-${mode} \
           --covar results/out/eur/hg19-${mode}.tcga-covariates.csv  \
           --pheno results/out/kegg-phenotypes.txt \
           --glm hide-covar --adjust --pfilter 1 \
           --out results/out/eur/hg19-${mode}
done

# Step 9 (section 2.2): Clumping (TODO: what are parameters); takes ~5 min
parallel "awk '(\$12 + 0) < 5.83E-9' {} > {}.top" ::: results/out/eur/hg19-*.linear
parallel "awk '(\$10 + 0) < 0.05'    {} > {}.top" ::: results/out/eur/hg19-*.adjusted
parallel --bar 'plink2 --pfile results/out/eur/hg19-qc --clump {} --clump-allow-overlap --clump-kb 250 --clump-p1 0.00000000583 --clump-p2 0.01 --clump-r2 0.2 --clump-snp-field ID --clump-field P --out {}.clump' ::: results/out/eur/hg19-qc.*.linear.top
parallel --bar 'plink2 --pfile results/out/eur/hg19-lowr2-qc --clump {} --clump-allow-overlap --clump-kb 250 --clump-p1 0.00000000583 --clump-p2 0.01 --clump-r2 0.2 --clump-snp-field ID --clump-field P --out {}.clump' ::: results/out/eur/hg19-lowr2-qc.*.linear.top
parallel --bar 'plink2 --pfile results/out/eur/hg19-nold-qc --clump {} --clump-allow-overlap --clump-kb 250 --clump-p1 0.00000000583 --clump-p2 0.01 --clump-r2 0.2 --clump-snp-field ID --clump-field P --out {}.clump' ::: results/out/eur/hg19-nold-qc.*.linear.top
parallel --bar 'plink2 --pfile results/out/eur/hg19-qc --clump {} --clump-allow-overlap --clump-kb 250 --clump-p1 0.00000000583 --clump-p2 0.01 --clump-r2 0.2 --clump-snp-field ID --clump-field BONF --out {}.clump' ::: results/out/eur/hg19-qc.*.adjusted.top
parallel --bar 'plink2 --pfile results/out/eur/hg19-lowr2-qc --clump {} --clump-allow-overlap --clump-kb 250 --clump-p1 0.00000000583 --clump-p2 0.01 --clump-r2 0.2 --clump-snp-field ID --clump-field BONF --out {}.clump' ::: results/out/eur/hg19-lowr2-qc.*.adjusted.top
parallel --bar 'plink2 --pfile results/out/eur/hg19-nold-qc --clump {} --clump-allow-overlap --clump-kb 250 --clump-p1 0.00000000583 --clump-p2 0.01 --clump-r2 0.2 --clump-snp-field ID --clump-field BONF --out {}.clump' ::: results/out/eur/hg19-nold-qc.*.adjusted.top

# Step 10: Final results
parallel "awk NF {} | awk '{ print \"{= s:results/out/eur/hg19-(.+)\.(hsa.+).glm.linear\.(.+).clump.clumps:\1-\2-\3: =}\",\$0 }' | sed 1d | tr '-' ' ' | awk '{print \$1,\$2,\$3,\$6,\$8,\$9}'" ::: results/out/eur/*.clumps | sort > results/out/top-eur.snps

# Gene-set analysis

In [ ]:
%%bash

# Step 1: SNP annotation using MAGMA; takes ~1 min
for mode in qc nold-qc lowr2-qc ; do
    plink2 --pfile results/out/eur/hg19-${mode} --make-bed --out results/out/eur/hg19-${mode}
    magma --annotate --snp-loc results/out/eur/hg19-${mode}.bim --gene-loc data/NCBI37.3.gene.loc --out results/out/eur/hg19-${mode}
done

# Step 2: Run MAGMA; takes ~1 hr 30 min per mode (~4 hrs 30 mins in total)
mkdir -p results/out/eur/magma
parallel --bar "awk '{print \$3,\$11,\$15}' {} > {}.pval" ::: results/out/eur/*.glm.linear
for mode in qc nold-qc lowr2-qc ; do
    parallel --bar "magma --bfile results/out/eur/hg19-${mode} --pval {} use=ID,P ncol=OBS_CT --gene-annot results/out/eur/hg19-${mode}.genes.annot --out results/out/eur/magma/{/.}.magma" ::: results/out/eur/hg19-${mode}.*.pval
done

# Step 3: Run A-LAVA; takes ~9 minutes per sample
for mode in qc nold-qc lowr2-qc ; do
    python alava.py out/kegg-pathways.txt results/out/eur/magma/hg19-${mode} > results/out/eur/hg19-${mode}.alava
done

# Plots and simulations

In [ ]:
# Step 1: simulations
mkdir -p results/out/plots
python code/simulate.py results/out results/out/eur/magma/hg19-qc data/simulations.txt results/out/plots

# Step 2: plots
python code/plots.py results/out results/out/eur/magma/hg19-qc results/out/plots

# Old stuff (not used/needed)

In [ ]:
%%bash

for f in out/magma-gene/*.genes.out
do
    sort -k9g $f > ${f}.sorted
    awk '{print $1}' ${f}.sorted > ${f}.entrez
    awk 'NR==FNR {A[$1]; next} ($2 in A)' ${f}.sorted out/entrez-ids.txt |\
        awk '{print $2, $1}' |\
        awk 'FNR == NR {lineno[$1] = NR; next} {print lineno[$1], $0}' <(sed 1d ${f}.entrez) - |\
        sort -k1,1g |\
        cut -d' ' -f2- |\
        awk 'NR==1 {print "Entrez_Id\tHGNC_Symbol"} {print}' > ${f}.hgnc
    paste -d'\t' ${f}.sorted ${f}.hgnc > ${f%.genes.out}.annotated
done
